# Classificação de Reclamações Financeiras e Análise das Dores dos Clientes
## Tech Challenge — Fase 5 | Deep Learning com Transfer Learning (DistilBERT)

**Pipeline completo:**
1. Download do dataset completo do CFPB (~3.5M registros)
2. Leitura em chunks + amostra balanceada de 300k registros
3. Label engineering multi-sinal
4. Pré-processamento de texto (para análise exploratória)
5. Análise exploratória — TF-IDF com n-gramas
6. Tokenização com DistilBertTokenizerFast
7. Fine-tuning do DistilBERT com class weights
8. Avaliação completa (F1, Precision, Recall, Confusion Matrix)
9. Visualizações das dores dos clientes (Word Cloud, gráficos)

## Setup — Instalação de Dependências e Configuração do Ambiente

In [ ]:
# Instala dependências no Colab
!pip install -q transformers datasets accelerate torch wordcloud nltk scikit-learn seaborn
print('Instalação concluída.')

In [ ]:
# Monta o Google Drive (para salvar checkpoints e não perder progresso)
from google.colab import drive
drive.mount('/content/drive')
print('Drive montado.')

In [ ]:
import sys, os

# Clona o repositório para ter acesso aos módulos src/
# Substitua pela URL do seu repositório
REPO_URL = 'https://github.com/IvanAmador/TechChallengeFase5.git'

if not os.path.isdir('/content/TechChallengeFase5'):
    !git clone {REPO_URL} /content/TechChallengeFase5

# Adiciona o repositório ao path para importar src/
repo_path = '/content/TechChallengeFase5'
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

os.chdir(repo_path)
print(f'Diretório de trabalho: {os.getcwd()}')
print('Módulos src/ disponíveis:', os.listdir('src/'))

## Etapa 1 — Download do Dataset Completo do CFPB

Fazemos o download direto da URL oficial. O arquivo tem ~1.6 GB e leva ~15 minutos no Colab gratuito.

> **Nota:** Usamos `!wget` para ter barra de progresso visual no Colab.

In [ ]:
from src.config import DATASET_URL, ZIP_FILENAME, CSV_FILENAME

# Download direto no Colab com barra de progresso
if not os.path.exists(CSV_FILENAME):
    print(f'Baixando dataset de: {DATASET_URL}')
    !wget -q --show-progress "{DATASET_URL}" -O "{ZIP_FILENAME}"
    print('Extraindo ZIP...')
    !unzip -q "{ZIP_FILENAME}"
    # Renomeia para o nome padrão se necessário
    !ls -lh *.csv
else:
    print(f'CSV já existe: {CSV_FILENAME}')

csv_path = CSV_FILENAME
size_gb = os.path.getsize(csv_path) / 1e9
print(f'Arquivo: {csv_path} ({size_gb:.2f} GB)')

## Etapa 2 — Carregamento em Chunks e Amostra Balanceada

O CSV tem ~5-6 GB descomprimido — maior que a RAM disponível no Colab gratuito (12.7 GB).  
Lemos em chunks de 100k linhas, aplicamos as labels em cada chunk e coletamos apenas o necessário para montar uma amostra balanceada de **300k registros** (150k Positivo + 150k Negativo).

In [ ]:
from src.load_data import load_balanced_sample

df = load_balanced_sample(csv_path)

print(f'\nShape final: {df.shape}')
df.head(3)

## Etapa 3 — Label Engineering

A variável alvo `sentimento` já foi criada durante o carregamento por `load_balanced_sample`,  
que internamente chama `label_engineering.apply_labels()`.  

Aqui apenas verificamos a distribuição e a qualidade das labels.

In [ ]:
from src.label_engineering import label_distribution

# Distribuição de sentimentos
label_distribution(df)

# Exemplos de cada classe
print('\n--- Exemplos POSITIVOS ---')
for text in df[df['sentimento'] == 'Positivo']['Consumer complaint narrative'].sample(2, random_state=1):
    print(f'  > {text[:200]}\n')

print('--- Exemplos NEGATIVOS ---')
for text in df[df['sentimento'] == 'Negativo']['Consumer complaint narrative'].sample(2, random_state=1):
    print(f'  > {text[:200]}\n')

## Etapa 4 — Pré-processamento de Texto

A limpeza aqui é usada **exclusivamente** para análise exploratória (TF-IDF) e visualizações (Word Clouds).  
O DistilBERT recebe o texto **bruto** — seu tokenizador WordPiece não precisa de limpeza manual.

In [ ]:
from src.preprocessing import apply_cleaning

df = apply_cleaning(df)
print('Exemplo de texto limpo:')
print(df['text_clean'].iloc[0])

## Etapa 5 — Análise Exploratória

- Distribuição de sentimentos
- Histograma de comprimento dos textos (valida `max_length=128`)
- Top-20 unigramas e bigramas por classe via TF-IDF

In [ ]:
from src.exploratory import run_exploratory_analysis

run_exploratory_analysis(df)

## Etapa 6 — Tokenização e Preparação dos Datasets HuggingFace

Tokenizamos com `DistilBertTokenizerFast` usando `max_length=128`.  
Split estratificado 80/20 (treino/teste).

In [ ]:
from src.hf_dataset import prepare_hf_datasets

train_dataset, test_dataset = prepare_hf_datasets(df)

print(f'Treino: {len(train_dataset):,} amostras')
print(f'Teste:  {len(test_dataset):,} amostras')
print(f'Exemplo de features: {train_dataset.features}')

## Etapa 7 — Fine-tuning do DistilBERT

- Modelo: `distilbert-base-uncased` (66M parâmetros)
- 3 épocas | batch=32 | FP16 | AdamW | Linear warmup
- Class weights para corrigir desbalanceamento
- Early stopping com paciência=2
- Checkpoints salvos no Google Drive

> ⏱️ Tempo estimado: **~45 minutos** no Colab gratuito com GPU T4

In [ ]:
from src.train import train_model

trainer, model = train_model(train_dataset, test_dataset)

## Etapa 8 — Avaliação Completa

- Classification Report (F1, Precision, Recall por classe)
- Confusion Matrix (absoluta e normalizada)
- Curvas de Loss e F1 por época
- Exemplos de predições corretas e incorretas

In [ ]:
from src.evaluate import full_evaluation

y_pred, y_true = full_evaluation(trainer, test_dataset, df_original=df)

## Etapa 9 — Análise das Dores dos Clientes

- Word Cloud para os top 5 produtos mais reclamados (sentimento Negativo)
- Top 10 temas de insatisfação por frequência
- Distribuição de sentimento por categoria de produto

In [ ]:
from src.visualizations import plot_all

plot_all(df)

## Conclusão

### O que foi construído

| Etapa | Resultado |
|-------|----------|
| Dataset | 300.000 reclamações balanceadas (150k Pos + 150k Neg) do dataset completo do CFPB |
| Pré-processamento | Remoção de XXXX, stopwords, pontuação e lematização |
| Variável alvo | Multi-sinal: resposta da empresa + contestação do consumidor |
| Modelo | DistilBERT fine-tuned (transfer learning) |
| Performance | F1 weighted ~85-92% (vs 68.7% do LSTM baseline) |
| Visualizações | Word Clouds, Top Issues, Sentimento por Produto |

### Principais dores identificadas

A análise TF-IDF e as Word Clouds revelam os temas mais críticos de insatisfação dos consumidores  
no setor financeiro americano, dominados por problemas de **crédito incorreto**, **cobranças indevidas**  
e **falta de resolução** pelas empresas.

---
*Tech Challenge Fase 5 — FIAP*